## Section 0 - Setup and Imports

In [ ]:
import sys
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

from utils.helpers import set_style, save_figure, start_logging
set_style()
log = start_logging(project_root, '03_04_model_comparison')

print('All imports successful')
log('All imports successful')

---
## Section 1 - Load All Forecast Data

In [ ]:
processed_path = project_root / 'data' / 'processed'

# Model metrics from Notebooks 2 and 3
metrics = pd.read_csv(processed_path / 'model_metrics.csv')

# Prophet forecast (full series)
prophet_fc = pd.read_csv(processed_path / 'prophet_forecast.csv',
                          parse_dates=['ds'])

# LightGBM forecast (test period only)
lgbm_fc = pd.read_csv(processed_path / 'lgbm_forecast.csv',
                       parse_dates=['week_start'])

# Full weekly feature data for context
weekly = pd.read_csv(processed_path / 'ts_features_weekly.csv',
                      parse_dates=['week_start'])

print('Data loaded')
print(f'Metrics table: {len(metrics)} models')
print(f'Prophet forecast: {len(prophet_fc):,} rows')
print(f'LightGBM forecast: {len(lgbm_fc):,} rows')
print()
print('Metrics summary:')
print(metrics.to_string(index=False))

log('Data loaded')
log(metrics.to_string())

## Section 2 - Consistent Comparison Framework


In [ ]:
# Align Prophet and LightGBM on a shared test window
# The LightGBM test period starts December 2020
# We extract the same weeks from Prophet for a direct comparison

LGBM_TEST_START = lgbm_fc['week_start'].min()
LGBM_TEST_END   = lgbm_fc['week_start'].max()

# Prophet forecast for the same window
prophet_test = prophet_fc[
    (prophet_fc['ds'] >= LGBM_TEST_START) &
    (prophet_fc['ds'] <= LGBM_TEST_END)
].copy()

print(f'Shared test window: {LGBM_TEST_START.date()} to {LGBM_TEST_END.date()}')
print(f'Prophet rows in window: {len(prophet_test)}')
print(f'LightGBM rows in window: {len(lgbm_fc)}')
print()

# Merge both forecasts on date for side-by-side comparison
# Rename columns to avoid confusion
prophet_test = prophet_test.rename(columns={
    'ds':             'week_start',
    'prophet_yhat':   'prophet_pred',
    'actual_revenue': 'actual'
})[['week_start', 'actual', 'prophet_pred']]

lgbm_test = lgbm_fc.rename(columns={
    'revenue':   'actual',
    'lgbm_pred': 'lgbm_pred'
})[['week_start', 'actual', 'lgbm_pred']]

comparison = prophet_test.merge(lgbm_test[['week_start', 'lgbm_pred']],
                                 on='week_start', how='inner')
comparison['prophet_pred'] = comparison['prophet_pred'].clip(lower=0)
comparison['lgbm_pred']    = comparison['lgbm_pred'].clip(lower=0)

# Per-week errors
comparison['prophet_error'] = comparison['actual'] - comparison['prophet_pred']
comparison['lgbm_error']    = comparison['actual'] - comparison['lgbm_pred']
comparison['prophet_abs_pct_error'] = np.abs(
    comparison['prophet_error'] / comparison['actual'].replace(0, np.nan)
) * 100
comparison['lgbm_abs_pct_error'] = np.abs(
    comparison['lgbm_error'] / comparison['actual'].replace(0, np.nan)
) * 100

print('Shared test period comparison:')
cols = ['week_start', 'actual', 'prophet_pred', 'lgbm_pred',
        'prophet_abs_pct_error', 'lgbm_abs_pct_error']
print(comparison[cols].round(1).to_string(index=False))

log('Shared test window comparison built')
log(comparison[cols].round(1).to_string())

In [ ]:
# Summary metrics on shared test window
def calc_mape(actual, predicted):
    mask = actual > 0
    return np.mean(np.abs(
        (actual[mask] - predicted[mask]) / actual[mask]
    )) * 100

shared_prophet_mape = calc_mape(
    comparison['actual'].values,
    comparison['prophet_pred'].values
)
shared_lgbm_mape = calc_mape(
    comparison['actual'].values,
    comparison['lgbm_pred'].values
)
shared_prophet_mae = np.mean(np.abs(comparison['prophet_error']))
shared_lgbm_mae    = np.mean(np.abs(comparison['lgbm_error']))

print('Metrics on shared test window')
print(f'  Prophet MAPE: {shared_prophet_mape:.1f}%')
print(f'  LightGBM MAPE: {shared_lgbm_mape:.1f}%')
print(f'  Prophet MAE: ${shared_prophet_mae:,.0f}/week')
print(f'  LightGBM MAE: ${shared_lgbm_mae:,.0f}/week')
print()

# Context: both models were trained on the same revenue level
# The test period is unreliable but we can still compare which
# model was less wrong on the shared window
if shared_lgbm_mape < shared_prophet_mape:
    winner = 'LightGBM'
    margin = shared_prophet_mape - shared_lgbm_mape
else:
    winner = 'Prophet'
    margin = shared_lgbm_mape - shared_prophet_mape

print(f'{winner} performed better on the shared test window by {margin:.1f} percentage points')

log(f'Shared window - Prophet MAPE: {shared_prophet_mape:.1f}%')
log(f'Shared window - LightGBM MAPE: {shared_lgbm_mape:.1f}%')


## Section 3 - Scorecard Comparison


In [ ]:
# Full scorecard comparison
prophet_row  = metrics[metrics['model'] == 'Prophet'].iloc[0]
lgbm_row     = metrics[metrics['model'] == 'LightGBM'].iloc[0]

print('Model comparison scorecard')
print()
print(f'{"Dimension":<35} {"Prophet":>20} {"LightGBM":>20}')
print('-' * 77)

rows = [
    ('Training MAPE',
     f"{prophet_row['train_mape']:.1f}%",
     f"{lgbm_row['train_mape']:.2f}% (overfit)"),
    ('Training weeks',
     str(int(prophet_row['train_weeks'])),
     str(int(lgbm_row['train_weeks']))),
    ('CV MAPE (honest estimate)',
     'Not computed',
     '11.6%'),
    ('Shared window MAPE',
     f'{shared_prophet_mape:.1f}%',
     f'{shared_lgbm_mape:.1f}%'),
    ('Shared window MAE',
     f'${shared_prophet_mae:,.0f}/wk',
     f'${shared_lgbm_mae:,.0f}/wk'),
    ('Handles change-points',
     'Yes (explicit)',
     'Yes (regime feature)'),
    ('Confidence intervals',
     'Yes (built-in)',
     'No'),
    ('Feature explainability',
     'Component decomposition',
     'SHAP values'),
    ('Setup complexity',
     'Low',
     'Medium'),
    ('Minimum data needed',
     '2 years recommended',
     '1 year minimum'),
    ('Overfitting risk (small data)',
     'Low',
     'High'),
]

for label, p_val, l_val in rows:
    print(f'{label:<35} {p_val:>20} {l_val:>20}')

log('Scorecard comparison printed')


## Section 4 - Visualisations

In [ ]:
# Chart 1: Side-by-side forecast comparison on shared test window
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(comparison['week_start'], comparison['actual'],
        color='#2E75B6', linewidth=2.5, marker='o', markersize=6,
        label='Actual revenue', zorder=4)
ax.plot(comparison['week_start'], comparison['prophet_pred'],
        color='#ED7D31', linewidth=1.8, marker='s', markersize=5,
        linestyle='--', label='Prophet forecast', zorder=3)
ax.plot(comparison['week_start'], comparison['lgbm_pred'],
        color='#70AD47', linewidth=1.8, marker='^', markersize=5,
        linestyle='--', label='LightGBM forecast', zorder=3)

ax.set_title('Prophet vs LightGBM - shared test window\n'
             f'Prophet MAPE: {shared_prophet_mape:.0f}%   '
             f'LightGBM MAPE: {shared_lgbm_mape:.0f}%',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Weekly revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=9)
plt.xticks(rotation=30, ha='right')

save_figure(fig, '03_04_model_comparison_forecast.png')
plt.show()

In [ ]:
# Chart 2: Per-week absolute percentage error comparison
# Shows which model was more accurate week by week

fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(comparison))
width = 0.35

bars1 = ax.bar(x - width/2, comparison['prophet_abs_pct_error'],
               width, color='#ED7D31', alpha=0.85, label='Prophet')
bars2 = ax.bar(x + width/2, comparison['lgbm_abs_pct_error'],
               width, color='#70AD47', alpha=0.85, label='LightGBM')

ax.set_xticks(x)
ax.set_xticklabels(
    [d.strftime('%b %d') for d in comparison['week_start']],
    rotation=30, ha='right'
)
ax.set_title('Absolute percentage error per week - Prophet vs LightGBM\n'
             'Lower bar = more accurate prediction for that week',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Absolute percentage error (%)')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

save_figure(fig, '03_04_error_comparison_by_week.png')
plt.show()

In [ ]:
# Chart 3: Training MAPE vs CV MAPE comparison
# Visualises the overfitting gap in LightGBM
# The gap between training and CV MAPE tells you how much
# the model is memorising vs genuinely learning

fig, ax = plt.subplots(figsize=(10, 5))

models       = ['Prophet', 'LightGBM']
train_mapes  = [prophet_row['train_mape'], lgbm_row['train_mape']]
cv_mapes_val = [25.13, 11.6]  # Prophet training = best honest estimate; LightGBM CV

x     = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, train_mapes, width,
               color='#2E75B6', alpha=0.85, label='Training MAPE')
bars2 = ax.bar(x + width/2, cv_mapes_val, width,
               color='#ED7D31', alpha=0.85,
               label='CV MAPE / honest estimate')

for bar, val in zip(bars1, train_mapes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
for bar, val in zip(bars2, cv_mapes_val):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_title('Training MAPE vs honest performance estimate\n'
             'Large gap between bars = model is overfitting to training data',
             fontsize=13, fontweight='medium')
ax.set_ylabel('MAPE (%)')
ax.legend(fontsize=9)

save_figure(fig, '03_04_training_vs_cv_mape.png')
plt.show()

In [ ]:
# Chart 4: Full timeline - actual revenue with both model forecasts
# Shows the complete picture from start of training to end of test

fig, ax = plt.subplots(figsize=(15, 5))

# Full actual revenue
ax.plot(weekly['week_start'], weekly['revenue'],
        color='#2E75B6', linewidth=1.5, label='Actual revenue', zorder=4)

# Prophet full forecast
ax.plot(prophet_fc['ds'], prophet_fc['prophet_yhat'].clip(lower=0),
        color='#ED7D31', linewidth=1.2, linestyle='--',
        alpha=0.8, label='Prophet forecast', zorder=3)

# LightGBM test period forecast
ax.plot(lgbm_fc['week_start'], lgbm_fc['lgbm_pred'],
        color='#70AD47', linewidth=2, linestyle='--',
        label='LightGBM forecast (test period)', zorder=3)

# Reference lines
ax.axvline(pd.to_datetime('2020-02-24'), color='#C00000',
           linestyle=':', linewidth=1.5, label='COVID change-point')
ax.axvline(pd.to_datetime('2020-11-30'), color='purple',
           linestyle=':', linewidth=1.5, label='LightGBM train/test split')

ax.set_title('Full timeline - actual revenue vs both model forecasts',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Weekly revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8, loc='upper left')
plt.xticks(rotation=30, ha='right')

save_figure(fig, '03_04_full_timeline_comparison.png')
plt.show()

---
## Section 5 - Findings and Recommendation

In [ ]:
print('Notebook 4 Findings - Model Comparison and Recommendation')
print()
print('Context')
print('  Both models were evaluated on a dataset with a known limitation:')
print('  the January-February 2021 test period contains a revenue collapse')
print('  caused by dataset truncation of pre-order orders. Test MAPE figures')
print('  for both models are therefore not meaningful in isolation.')
print('  This comparison uses training MAPE, cross-validation, and a')
print('  shared test window analysis to reach a fair conclusion.')
print()
print('Finding 1 - LightGBM has a better honest performance estimate')
print('  LightGBM cross-validation MAPE: 11.6%')
print('  Prophet training MAPE: 25.1% (best available honest estimate')
print('  since Prophet cross-validation was not computed)')
print('  On the shared test window, LightGBM also performed better:')
print(f'  LightGBM MAPE {shared_lgbm_mape:.0f}% vs Prophet MAPE {shared_prophet_mape:.0f}%')
print()
print('Finding 2 - LightGBM overfits severely on small datasets')
print('  LightGBM training MAPE of 0.83% vs CV MAPE of 11.6% reveals')
print('  a large overfitting gap. The model memorised 40 training weeks')
print('  rather than learning robust patterns. This is a direct result')
print('  of having 22 features on only 40 data points.')
print()
print('Finding 3 - Prophet is more appropriate for small datasets')
print('  Prophet uses Bayesian inference rather than fitting to data points')
print('  directly. It incorporates prior knowledge about trend and seasonality')
print('  which acts as regularisation. This makes it less likely to overfit')
print('  on limited data than a tree-based model like LightGBM.')
print()
print('Finding 4 - Both models identified the same structural patterns')
print('  Prophet component decomposition and LightGBM SHAP both found:')
print('  - A clear monthly seasonality signal')
print('  - Product mix (high-ticket percentage) as a key revenue driver')
print('  - Recent revenue momentum as a core predictor')
print('  - No meaningful weekend or day-of-week effect')
print('  Convergence across two different methodologies strengthens')
print('  confidence in these findings.')
print()
print('Finding 5 - The dataset is the binding constraint')
print('  Both models are limited by 2 years of data, a mid-period')
print('  structural break, and a truncated endpoint. Neither model')
print('  can produce production-quality forecasts from this data.')
print('  A 3-5 year dataset with consistent business conditions would')
print('  allow both models to demonstrate their full capability.')
print()
print('Finding 6 - LightGBM is more adaptive to sudden revenue changes')
print('  When actual revenue fell sharply in January 2021, LightGBM')
print('  lag features updated with the falling values each week,')
print('  pulling forecasts downward. Prophet continued projecting')
print('  its trained upward trend regardless of new signal.')
print('  On the shared test window LightGBM outperformed Prophet')
print(f'  by {shared_prophet_mape - shared_lgbm_mape:.0f} percentage points (MAPE {shared_lgbm_mape:.0f}% vs {shared_prophet_mape:.0f}%).')
print('  In a production setting where revenue can shift suddenly,')
print('  this adaptive behaviour is a meaningful advantage.')
print()
print('Recommendation')
print()
print('  For this dataset: use Prophet')
print('  Reason: lower overfitting risk on small data, built-in')
print('  confidence intervals, interpretable component decomposition,')
print('  and explicit change-point handling without requiring manually')
print('  engineered lag and rolling features.')
print()
print('  For a richer dataset (3+ years, stable conditions): use LightGBM')
print('  Reason: with sufficient data to prevent overfitting, LightGBM')
print('  can capture non-linear interactions between features that Prophet')
print('  cannot. The SHAP explainability also provides richer business')
print('  insight than Prophets component decomposition.')
print()
print('  In production: run both in parallel and use an ensemble')
print('  A weighted average of Prophet and LightGBM forecasts typically')
print('  outperforms either model alone by reducing individual model error.')


## Section 6 - Export Summary

In [ ]:
# Save the comparison table for the Excel report
comparison_export = comparison[[
    'week_start', 'actual', 'prophet_pred', 'lgbm_pred',
    'prophet_error', 'lgbm_error',
    'prophet_abs_pct_error', 'lgbm_abs_pct_error'
]].copy()

output_path = processed_path / 'model_comparison.csv'
comparison_export.to_csv(output_path, index=False)

print(f'Exported: model_comparison.csv ({len(comparison_export):,} rows)')